<h1>现代 Hopfield 网络：MNIST Bags 多实例学习实验</h1>

## Colab 环境

请先运行下面的代码单元。它会安装固定版本的 `hflayers`，下载固定版本的 ADMIL 基线模型，并导入后续实验所需的全部依赖。安装与导入放在同一个单元中，因此环境问题会在实验开始前直接暴露。

In [ ]:
%cd /content
%pip install -q "git+https://github.com/ml-jku/hopfield-layers.git@f56f929c95b77a070ae675ea4f56b6d54d36e730"
!apt-get install -qq fonts-noto-cjk > /dev/null
!mkdir -p /content/nn-labs-deps
!rm -rf /content/nn-labs-deps/AttentionDeepMIL
!git clone -q https://github.com/AMLab-Amsterdam/AttentionDeepMIL.git /content/nn-labs-deps/AttentionDeepMIL
!git -C /content/nn-labs-deps/AttentionDeepMIL checkout -q eb0434ba2795711a45d693d60120ae53532b1b93

# 导入绘图与数据处理所需的通用模块。
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import sys
import torch

# 导入现代 Hopfield pooling。
from hflayers import HopfieldPooling

# 导入版本检查与类型标注工具。
from packaging.version import Version
from typing import Optional, Tuple

# 导入 PyTorch 训练所需模块。
from torch import Tensor
from torch.nn import Conv2d, Dropout, Linear, MaxPool2d, Module, ReLU, Sequential, Sigmoid
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

# 配置支持中文的绘图字体与样式。
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(font='Noto Sans CJK SC')
print(f'hflayers 导入成功：{HopfieldPooling.__module__}')

## 官方基线依赖

将 Attention-based Deep Multiple Instance Learning（ADMIL）仓库加入 Python 模块搜索路径。实验从该仓库导入官方的 `Attention` 与 `GatedAttention` 基线；旧版数据加载器则由下方兼容当前 PyTorch 的实现替代。

In [ ]:
sys.path.insert(0, r'/content/nn-labs-deps/AttentionDeepMIL')

## 数据加载器

导入官方基线模型，并定义当前 PyTorch 可用的 `MnistBags`。该实现保留原始抽样规则，每个 bag 单独保存为变长张量，不会把长度不同的 bag 强行堆叠成同一个张量。

In [ ]:
from model import Attention, GatedAttention


class MnistBags(Dataset):
    """按照原始 ADMIL 抽样规则构造变长 MNIST bags。"""

    def __init__(self, target_number=9, mean_bag_length=10, var_bag_length=2,
                 num_bag=250, seed=1, train=True, root='/content/datasets'):
        self.target_number = target_number
        self.mean_bag_length = mean_bag_length
        self.var_bag_length = var_bag_length
        self.num_bag = num_bag
        self.rng = np.random.RandomState(seed)
        self.mnist = datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]),
        )
        self.bags, self.instance_labels = self._create_bags()

    def _create_bags(self):
        bags, instance_labels = [], []
        for _ in range(self.num_bag):
            bag_length = max(1, int(self.rng.normal(
                self.mean_bag_length, self.var_bag_length, size=1
            ).item()))
            indices = self.rng.randint(0, len(self.mnist), size=bag_length)
            instances = [self.mnist[int(index)] for index in indices]

            bags.append(torch.stack([image for image, _ in instances]))
            instance_labels.append(torch.tensor(
                [label == self.target_number for _, label in instances],
                dtype=torch.bool,
            ))

        return bags, instance_labels

    def __len__(self):
        return self.num_bag

    def __getitem__(self, index):
        labels = self.instance_labels[index]
        return self.bags[index], (labels.any(), labels)

## 环境检查

检查当前 Python 与 PyTorch 版本是否满足实验的最低要求。

In [ ]:
python_check = '(\u2713)' if sys.version_info >= (3, 8) else '(\u2717)'
pytorch_check = '(\u2713)' if Version(torch.__version__.split('+')[0]) >= Version(r'1.5') else '(\u2717)'

print(f'Python 版本： {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro} {python_check}')
print(f'PyTorch 版本：{torch.__version__} {pytorch_check}')

## 实验定位

这份 notebook 研究一个任务，并在完全相同的数据上训练三种模型：`Attention`、`GatedAttention` 与 `HopfieldPooling`。前两种是多实例学习基线，第三种是论文配套仓库要展示的现代 Hopfield pooling。三张结果图不是三个数据集，而是三种模型各自的训练结果。

## 目标定义

这是有监督学习中的二分类与判别任务，同时属于多实例学习（Multiple Instance Learning，MIL）。一个样本不是单张图片，而是由若干图片组成的 bag。bag 中只要出现至少一个数字 `9`，bag 标签就是正类；否则就是负类。模型只获得 bag 级标签，不获得哪张图片是 `9` 的实例级监督。

输入是形状近似为 `bag 长度 × 1 × 28 × 28` 的变长图片集合，输出是该 bag 属于正类的概率以及阈值化后的二分类结果。这个实验不涉及生成、聚类、强化学习或因果推断。

## 数据与预处理

底层数据来自 MNIST。每张灰度图先转换为张量，再用 MNIST 的均值 `0.1307` 与标准差 `0.3081` 做标准化。bag 长度从以 `10` 为中心、标准差为 `2` 的正态分布中抽样，且至少包含一张图片。

这里没有手工设计图像特征。卷积网络直接从像素学习每张图片的表示，因此采用的是端到端表征学习。bag 内图片被视为集合，任务不依赖图片排列顺序。

## 数据集划分

训练数据由 MNIST 训练集抽样生成 `200` 个 bag，评估数据由 MNIST 测试集抽样生成 `50` 个 bag。代码每个 epoch 都查看评估集，因此它实际上承担验证集作用。实验没有再保留一个完全独立、只在最后使用一次的 held-out 测试集。

## 模型视角

从数据结构看，输入是变长集合；从关系语义看，模型学习的是图片特征与 bag 标签之间的关联，而不是因果关系；从数学对象看，整体模型是一个判别映射，内部的现代 Hopfield 层还可以从能量函数与联想检索的角度理解。

MNIST Bags 任务来自 Ilse、Tomczak 与 Welling 的 Attention-based Deep Multiple Instance Learning 工作。当前示例在其 Attention 架构基础上加入 Hopfield pooling，用来观察不同聚合机制能否找出决定 bag 标签的关键实例。

<table>
    <tr><th>参数</th><th>实验值</th><th>含义</th></tr>
    <tr><td><code>target_number</code></td><td>9</td><td>决定正类的目标数字</td></tr>
    <tr><td><code>mean_bag_length</code></td><td>10</td><td>bag 长度的抽样中心</td></tr>
    <tr><td><code>var_bag_length</code></td><td>2</td><td>代码沿用的参数名，实际作为正态分布标准差传入</td></tr>
    <tr><td><code>num_bag</code></td><td>训练 200，评估 50</td><td>训练与评估样本量</td></tr>
</table>

In [ ]:
device = torch.device(r'cuda:0' if torch.cuda.is_available() else r'cpu')

# 从 MNIST 训练集抽样生成训练 bags。
data_loader_train = DataLoader(MnistBags(
    target_number=9,
    mean_bag_length=10,
    var_bag_length=2,
    num_bag=200,
    train=True
), batch_size=1, shuffle=True)

# 从 MNIST 测试集抽样生成评估 bags。
data_loader_eval = DataLoader(MnistBags(
    target_number=9,
    mean_bag_length=10,
    var_bag_length=2,
    num_bag=50,
    train=False
), batch_size=1, shuffle=True)

In [ ]:
log_dir = f'resources/'
os.makedirs(log_dir, exist_ok=True)

## 损失、优化与训练

三种模型都输出正类概率，并使用二元交叉熵衡量预测概率与 bag 标签之间的差异。优化器使用 `AdamW`，学习率为 `5e-4`，权重衰减为 `1e-4`，反向传播后将梯度范数裁剪到 `1.0`。

每种模型都从随机初始化开始训练 `20` 个 epoch。这不是微调，也不使用 LoRA 或其他参数高效微调方法。LoRA 主要适用于已经预训练的大模型；当前实验的 CNN、聚合层与分类器参数量较小，直接从头训练更合适。

训练阶段使用 `train()`，每个 bag 计算损失、反向传播并更新参数。评估阶段使用 `eval()` 与 `torch.no_grad()`，只计算指标而不更新模型。三种模型在训练前都重置随机种子，便于进行较公平的单次比较。

## 验证与超参数调优

每个 epoch 后都会在评估 bag 上记录 loss、error 与 accuracy。`error` 等于 `1 - accuracy`；accuracy 只判断类别是否正确，二元交叉熵还会考虑概率置信度，因此两条曲线不一定同步变化。

当前超参数由官方示例固定，没有执行网格搜索、随机搜索或贝叶斯优化，也没有 early stopping 与最佳 checkpoint 保存。由于只有一个随机种子和 `50` 个评估 bag，曲线适合观察模型是否学会任务，但不足以证明某个模型稳定优于其他模型。

## 测试集评估边界

这里没有独立的最终测试阶段，也没有交叉验证。若根据评估曲线选择 epoch，这组数据就应被视为验证集，不能再作为无偏的最终测试结果。因此本 notebook 是一个机器学习研究实验的最小闭环，不是包含数据治理、系统调参、最终测试、部署和监控的完整工业流程。

In [ ]:
def train_epoch(network: Module,
                optimiser: AdamW,
                data_loader: DataLoader
               ) -> Tuple[float, float, float]:
    """执行一个训练 epoch，返回平均损失、错误率与准确率。"""
    network.train()
    losses, errors, accuracies = [], [], []
    for data, target in data_loader:
        data, target = data.to(device=device), target[0].to(device=device)

        # 前向传播并计算当前 bag 的目标函数。
        loss = network.calculate_objective(data, target)[0]

        # 反向传播、裁剪梯度并更新模型参数。
        optimiser.zero_grad()
        loss.backward()
        clip_grad_norm_(parameters=network.parameters(), max_norm=1.0, norm_type=2)
        optimiser.step()

        # 记录参数更新后的分类表现。
        error, prediction = network.calculate_classification_error(data, target)
        accuracy = (prediction == target).to(dtype=torch.float32).mean()
        accuracies.append(accuracy.detach().item())
        errors.append(error)
        losses.append(loss.detach().item())
    
    # 返回整个 epoch 的平均指标。
    return sum(losses) / len(losses), sum(errors) / len(errors), sum(accuracies) / len(accuracies)


def eval_iter(network: Module,
              data_loader: DataLoader
             ) -> Tuple[float, float, float]:
    """在不更新参数的情况下评估模型，返回平均损失、错误率与准确率。"""
    network.eval()
    with torch.no_grad():
        losses, errors, accuracies = [], [], []
        for data, target in data_loader:
            data, target = data.to(device=device), target[0].to(device=device)

            # 前向传播并计算当前 bag 的目标函数。
            loss = network.calculate_objective(data, target)[0]

            # 记录当前模型的分类表现。
            error, prediction = network.calculate_classification_error(data, target)
            accuracy = (prediction == target).to(dtype=torch.float32).mean()
            accuracies.append(accuracy.detach().item())
            errors.append(error)
            losses.append(loss.detach().item())

        # 返回全部评估 bags 的平均指标。
        return sum(losses) / len(losses), sum(errors) / len(errors), sum(accuracies) / len(accuracies)

    
def operate(network: Module,
            optimiser: AdamW,
            data_loader_train: DataLoader,
            data_loader_eval: DataLoader,
            num_epochs: int = 1
           ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """使用反向传播训练指定模型，并保存每个 epoch 的训练与评估指标。"""
    losses, errors, accuracies = {r'train': [], r'eval': []}, {r'train': [], r'eval': []}, {r'train': [], r'eval': []}
    for epoch in range(num_epochs):
        
        # 执行训练阶段。
        performance = train_epoch(network, optimiser, data_loader_train)
        losses[r'train'].append(performance[0])
        errors[r'train'].append(performance[1])
        accuracies[r'train'].append(performance[2])
        
        # 执行评估阶段。
        performance = eval_iter(network, data_loader_eval)
        losses[r'eval'].append(performance[0])
        errors[r'eval'].append(performance[1])
        accuracies[r'eval'].append(performance[2])
    
    # 将完整训练历史转换为便于绘图的数据表。
    return pd.DataFrame(losses), pd.DataFrame(errors), pd.DataFrame(accuracies)

In [ ]:
def set_seed(seed: int = 42) -> None:
    """设置 PyTorch 随机种子，并启用确定性 cuDNN 行为。"""
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def plot_performance(loss: pd.DataFrame,
                     error: pd.DataFrame,
                     accuracy: pd.DataFrame,
                     log_file: str,
                     experiment_name: str
                    ) -> None:
    """绘制并保存训练与评估阶段的损失、错误率和准确率。"""
    fig, ax = plt.subplots(1, 3, figsize=(20, 7))
    
    column_names = {r'train': r'训练', r'eval': r'评估'}
    loss_plot = sns.lineplot(data=loss.rename(columns=column_names), ax=ax[0])
    loss_plot.set(xlabel=r'训练轮次', ylabel=r'损失')
    
    error_plot = sns.lineplot(data=error.rename(columns=column_names), ax=ax[1])
    error_plot.set(xlabel=r'训练轮次', ylabel=r'错误率')
    
    accuracy_plot = sns.lineplot(data=accuracy.rename(columns=column_names), ax=ax[2])
    accuracy_plot.set(xlabel=r'训练轮次', ylabel=r'准确率')
    fig.suptitle(experiment_name, fontsize=16)
    
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(log_file)
    plt.show(fig)

## Attention 基线

卷积网络先把 bag 中每张图片编码成特征向量。Attention 模块为每个实例计算权重，再对全部实例做加权求和，最后通过 sigmoid 分类器输出 bag 属于正类的概率。

In [ ]:
set_seed()
network = Attention().to(device=device)
optimiser = AdamW(params=network.parameters(), lr=5e-4, weight_decay=1e-4)

### 训练并观察 Attention 基线

In [ ]:
losses, errors, accuracies = operate(
    network=network,
    optimiser=optimiser,
    data_loader_train=data_loader_train,
    data_loader_eval=data_loader_eval,
    num_epochs=20)

In [ ]:
plot_performance(
    loss=losses, error=errors, accuracy=accuracies,
    log_file=f'{log_dir}/attention_base.pdf', experiment_name='Attention 基线'
)

## GatedAttention 基线

GatedAttention 与普通 Attention 使用相同的卷积特征提取器，但计算权重时增加一条 sigmoid 门控分支，并与 tanh 分支逐元素相乘。这个基线用于观察更灵活的注意力打分函数是否改善 bag 聚合。

In [ ]:
set_seed()
network = GatedAttention().to(device=device)
optimiser = AdamW(params=network.parameters(), lr=5e-4, weight_decay=1e-4)

### 训练并观察 GatedAttention 基线

In [ ]:
losses, errors, accuracies = operate(
    network=network,
    optimiser=optimiser,
    data_loader_train=data_loader_train,
    data_loader_eval=data_loader_eval,
    num_epochs=20)

In [ ]:
plot_performance(
    loss=losses, error=errors, accuracy=accuracies,
    log_file=f'{log_dir}/gated_attention_base.pdf', experiment_name='GatedAttention 基线'
)

## HopfieldPooling 实验

该模型沿用相同的卷积特征提取器，把每张图片编码为 `500` 维向量，再交给 `HopfieldPooling` 聚合整个 bag。Hopfield pooling 的隐藏维度为 `32`，使用一个 head，输出仍为 `500` 维；聚合结果经过 dropout 与 sigmoid 分类器得到 bag 概率。

现代 Hopfield 层在这里不是逐城市求解旅行商问题，而是执行连续状态下的联想与聚合：它根据当前可学习的查询，从多个实例特征中集中提取对 bag 分类最有用的信息。

In [ ]:
class HfPooling(Module):    
    def __init__(self):
        """初始化 Hopfield pooling 网络。超参数沿用官方演示设置。"""
        super(HfPooling, self).__init__()
        self.L = 500
        self.D = 128
        self.K = 1

        self.feature_extractor_part1 = Sequential(
            Conv2d(1, 20, kernel_size=5),
            ReLU(),
            MaxPool2d(2, stride=2),
            Conv2d(20, 50, kernel_size=5),
            ReLU(),
            MaxPool2d(2, stride=2)
        )
        self.feature_extractor_part2 = Sequential(
            Linear(50 * 4 * 4, self.L),
            ReLU(),
        )
        self.hopfield_pooling = HopfieldPooling(
            input_size=self.L, hidden_size=32, output_size=self.L, num_heads=1
        )
        self.dp = Dropout(
            p=0.1
        )
        self.classifier = Sequential(
            Linear(self.L * self.K, 1),
            Sigmoid()
        )
        
    def forward(self, input: Tensor) -> Tuple[Tensor, Tensor, Optional[Tensor]]:
        """提取每个实例的特征，聚合整个 bag，并输出分类概率。"""
        x = input.squeeze(0)
        H = self.feature_extractor_part1(x)
        H = H.view(-1, 50 * 4 * 4)
        H = self.feature_extractor_part2(H)
        
        H = H.unsqueeze(0)
        H = self.hopfield_pooling(H)
        H = H.squeeze(0)
        H = self.dp(H)

        Y_prob = self.classifier(H)
        Y_hat = torch.ge(Y_prob, 0.5).float()

        return Y_prob, Y_hat, None

    def calculate_classification_error(self, input: Tensor, target: Tensor) -> Tuple[Tensor, Tensor]:
        """计算当前模型的分类错误率，并返回预测类别。"""
        Y = target.float()
        _, Y_hat, _ = self.forward(input)
        error = 1.0 - Y_hat.eq(Y).cpu().float().mean().item()

        return error, Y_hat

    def calculate_objective(self, input: Tensor, target: Tensor) -> Tuple[Tensor, Optional[Tensor]]:
        """计算二元负对数似然；第二个返回值保留官方接口中的占位项。"""
        Y = target.float()
        Y_prob, _, A = self.forward(input)
        Y_prob = torch.clamp(Y_prob, min=1e-5, max=(1.0 - 1e-5))
        neg_log_likelihood = -1.0 * (Y * torch.log(Y_prob) + (1.0 - Y) * torch.log(1.0 - Y_prob))

        return neg_log_likelihood, A

In [ ]:
set_seed()
network = HfPooling().to(device=device)
optimiser = AdamW(params=network.parameters(), lr=5e-4, weight_decay=1e-4)

### 训练并观察 HopfieldPooling

In [ ]:
losses, errors, accuracies = operate(
    network=network,
    optimiser=optimiser,
    data_loader_train=data_loader_train,
    data_loader_eval=data_loader_eval,
    num_epochs=20)

In [ ]:
plot_performance(
    loss=losses, error=errors, accuracy=accuracies,
    log_file=f'{log_dir}/hopfield_pooling.pdf', experiment_name='HopfieldPooling 实验'
)

## 如何比较三组结果

三组实验使用同一任务、同一训练与评估数据生成规则、同一训练轮数和同一优化器，因此可以先比较它们是否成功学习，再比较评估集表现与训练稳定性。

阅读曲线时应同时看训练与评估。训练 loss 下降、训练 accuracy 接近 `1.0` 只能说明模型拟合了训练 bag；评估 accuracy 明显高于随机猜测才说明模型对新 bag 具有一定泛化能力。若训练继续改善而评估 loss 上升或 accuracy 下降，则出现了过拟合。评估集只有 `50` 个 bag，每错一个样本就会让 accuracy 改变两个百分点，因此橙色曲线本身会比较抖动。

二元交叉熵关心预测概率的置信度，accuracy 只关心概率经过 `0.5` 阈值后是否分对。因此少数非常自信的错误预测可能显著抬高评估 loss，而评估 accuracy 仍保持较高。

这份 notebook 可以支持的结论是三种聚合机制能否在该次运行中学会 MNIST Bags 任务。若要严谨声称 HopfieldPooling 优于基线，还需要多个随机种子、更大的验证与测试集、最佳 checkpoint 规则，以及均值、方差或置信区间。